In [ ]:
import os
import re
import json
import time

from pathlib import Path
from collections import deque, defaultdict
from semanticher.onto import OntologyDAG
from semanticher.data import load_table_list, load_tables, load_label_class

from langchain_openai.chat_models.base import BaseChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# Base folders
REPO_ROOT       = Path().resolve().parents[1]
BASE_RESULT_DIR = REPO_ROOT / "results" / "main_edm"
RESULT_DIR      = BASE_RESULT_DIR / "Result_edm"
LOG_DIR         = BASE_RESULT_DIR / "Promptinput_edm"
SUMMARY_DIR     = BASE_RESULT_DIR / "Tokenusage_Duration_edm"

os.makedirs(RESULT_DIR,  exist_ok=True)
os.makedirs(LOG_DIR,     exist_ok=True)
os.makedirs(SUMMARY_DIR, exist_ok=True)

# build DAG
dag = OntologyDAG()
dag.build_dag()

n_level1, n_level2 = 0, 0
for o in dag.edges.get(dag.root, []):
    n_level1 += 1
    print(dag.name_of_uri(o))
    for c in dag.edges.get(o, []):
        n_level2 += 1
        print("\t", dag.name_of_uri(c))
print(f"Number of level 1: {n_level1}")
print(f"Number of level 2: {n_level2}")

df_table_list = load_table_list()
df_labels     = load_label_class()
dict_tables   = load_tables()

k = 5
max_depth = 2


def dataframe_to_markdown(df, k=5):
    subset  = df.head(k)
    headers = "| " + " | ".join(subset.columns) + " |"
    sep     = "| " + " | ".join(["---"] * len(subset.columns)) + " |"

    rows = []
    for _, row in subset.iterrows():
        row_str = "| " + " | ".join(map(str, row.values)) + " |"
        rows.append(row_str)

    return headers + "\n" + sep + "\n" + "\n".join(rows)


# =========================
# Model aliases / experiments
# =========================
MODEL_ALIAS = {
    "deepseek-chat":    "dsk",
    "gemini-2.0-flash": "gem",
    "gpt-4.1-mini":     "gpt",
}

# threshold per number of agents  (single/three → 0.66, two → 0.50)
THRESHOLD = {1: 0.66, 2: 0.50, 3: 0.66}

EXPERIMENTS = [
    ["deepseek-chat"],
    ["gemini-2.0-flash"],
    ["gpt-4.1-mini"],
    ["deepseek-chat", "gemini-2.0-flash"],
    ["deepseek-chat", "gpt-4.1-mini"],
    ["gemini-2.0-flash", "gpt-4.1-mini"],
    ["deepseek-chat", "gemini-2.0-flash", "gpt-4.1-mini"],
]


def build_experiment_tag(model_names):
    aliases = [MODEL_ALIAS[m] for m in model_names]
    prefix  = {1: "single", 2: "two", 3: "three"}[len(model_names)]
    return f"{prefix}_{'_'.join(aliases)}"


def build_result_path(model_names):
    tag = build_experiment_tag(model_names)
    return os.path.join(RESULT_DIR,  f"edm_{tag}.json")


def build_logs_path(model_names):
    tag = build_experiment_tag(model_names)
    return os.path.join(LOG_DIR,     f"logs_edm_{tag}.json")


def build_summary_path(model_names):
    tag = build_experiment_tag(model_names)
    return os.path.join(SUMMARY_DIR, f"summary_edm_{tag}.json")

OccupantBehavior
PowerSystemResource
	 EquipmentContainer
	 PowerEquipment
PowerDeliveryUnit
District
KeyPerformanceIndicator
	 StrategicKPI
	 TacticalKPI
	 OperationalKPI
PerformanceGoal
KPICalculation
KPIEvaluatedObject
KPICalculationComponent
	 Assumption
	 Mathematical_model
	 UniversalConstant
	 Variable
	 Equation
	 DatumSource
Stakeholder
Unit_of_measure
	 Compound_unit
	 Singular_unit
KPIValue
ObservationValue
Observation
Schedule
FeatureOfInterest
ObservationProperty
	 EquipmentParameter
	 OccupancyParameter
	 EnergyParameter
	 EnvironmentalParameter
	 BuildingParameter
Slot
Ordered List
	 Route
Event
Location
GeographicalCoordinatesPoint
PhysicalObject
	 Building1
	 Occupant
	 Equipment
	 BuildingThing
Certificate
Observation
Observable Property
AC Meausrement
Property
	 Energy
	 Power
	 Power
	 Flow
	 State of charge
	 CO2 level
	 Current1
	 Voltage1
	 Tap position
	 Volume
	 AC Frequency
	 Alarm2
	 Current2
	 Day Type
	 Water Consumption
	 Gas Consumption
	 Heat Consumption

* Owlready2 * WARNING: DataProperty http://www.owl-ontologies.com/EPC4EU#heatingDegreeDays belongs to more than one entity types: [owl.ObjectProperty, owl.DatatypeProperty]; I'm trying to fix it...
* Owlready2 * WARNING: DataProperty https://smartdatamodels.org/owner belongs to more than one entity types: [owl.ObjectProperty, owl.DatatypeProperty]; I'm trying to fix it...
* Owlready2 * WARNING: DataProperty http://purl.org/dc/terms/description belongs to more than one entity types: [owl.AnnotationProperty, owl.DatatypeProperty]; I'm trying to fix it...
* Owlready2 * WARNING: DataProperty http://purl.org/dc/terms/title belongs to more than one entity types: [owl.AnnotationProperty, owl.DatatypeProperty, owl.FunctionalProperty]; I'm trying to fix it...


In [ ]:
# =========================
# LLMs
# =========================
llm1 = BaseChatOpenAI(
    model='deepseek-chat', 
    openai_api_key='', 
    openai_api_base='https://api.deepseek.com',
    temperature=0,
    max_tokens=8192
)

llm2 = BaseChatOpenAI(
    model='gemini-2.0-flash', 
    openai_api_key='', 
    openai_api_base='https://generativelanguage.googleapis.com/v1beta/openai',
    temperature=0,
    max_tokens=None
)

llm3 = BaseChatOpenAI(
    model='gpt-4.1-mini', 
    openai_api_key='', 
    openai_api_base='https://api.openai.com/v1',
    temperature=0,
    max_tokens=None
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a knowledgeable assistant who helps map tabular data columns to ontology classes. You have expert knowledge in semantic table annotation, i.e., table column-to-ontology class mappings. Your goal is to determine the most semantically appropriate ontology class (or set of classes) for a given column, based on the provided table name, table header, example data, column name, and the candidate ontology classes."
        ),
        (
            "human",
            """
            We have a table named '{table_name}' which has several columns. Below is the table in Markdown format, including column headers and a few example rows:

            {table_in_markdown}

            We are currently focusing on ontology classes at the given level. Below are a subset of the ontology classes available at this level:
            {current_level_ontology_classes}

            We want to determine the best fitting ontology class (or class path if multiple levels are considered) for the following column: '{column_name}'.

            Instructions:
                1. Review the column name and the example data.
                2. Based ONLY on the ontology class names provided (at this level), select the most suitable ontology class name that best describes the semantic meaning of this column.
                3. Output the final answer enclosed in <answer></answer> tags.
                3.1 If multiple ontology class NAMES are required, split them with a comma, e.g. <answer>ClassName1, ClassName2</answer>
                3.2 If no suitable class is found, respond with <answer>-</answer>.
                Important:
                - You MUST choose only from the provided ontology class names at this level.
            """
        ),
    ]
)

CHAIN_REGISTRY = {
    "deepseek-chat":    prompt | llm1,
    "gemini-2.0-flash": prompt | llm2,
    "gpt-4.1-mini":     prompt | llm3,
}

In [4]:
def ensemble_decision_making(
    table_name,
    table_in_markdown,
    column_name,
    current_level_ontology_classes,
    chains,
    model_names,
    num_agents,
    consensus_threshold_ratio,
):
    """
    Ensemble decision making with LLM-based agents.
      - current_level_ontology_classes is treated as a list of NAMES (strings).
      - We deduplicate names per level so LLM sees each name only once.
      - LLM is expected to output NAMES inside <answer>...</answer>.
    """
    classes = [c.strip() for c in current_level_ontology_classes if isinstance(c, str) and c.strip()]
    if not classes:
        return "-", []

    classes       = sorted(set(classes))
    agent_classes = list(classes)
    valid_classes = set(classes)

    votes_per_class = defaultdict(int)

    def extract_answer(result):
        answer_pattern = r"<answer>(.*?)</answer>"
        answer_matches = re.findall(answer_pattern, result.content, flags=re.DOTALL)
        if not answer_matches:
            return []

        raw = answer_matches[0].strip()
        if raw == "-" or not raw:
            return []

        picked = [x.strip() for x in raw.split(",") if x.strip()]
        return [x for x in picked if x in valid_classes]

    records = []
    for agent_id in range(num_agents):
        assigned_classes_str = ", ".join(agent_classes)
        input_dict = {
            "table_name":                    table_name,
            "table_in_markdown":             table_in_markdown,
            "column_name":                   column_name,
            "current_level_ontology_classes": assigned_classes_str,
        }

        model_name      = model_names[agent_id]
        prompt_template = chains[agent_id].first
        formatted_prompt = prompt_template.format(**input_dict)

        if isinstance(formatted_prompt, str):
            prompt_text = formatted_prompt.strip()
        else:
            messages    = formatted_prompt.to_messages()
            prompt_text = "\n".join([f"[{m.type.upper()}]\n{m.content}" for m in messages])

        result      = chains[agent_id].invoke(input_dict)
        result_text = result.content

        usage_raw   = result.response_metadata.get("token_usage", {})
        usage_clean = {
            "input_tokens":  usage_raw.get("prompt_tokens", 0),
            "output_tokens": usage_raw.get("completion_tokens", 0),
            "total_tokens":  usage_raw.get("total_tokens", 0),
        }

        records.append({
            "model":  model_name,
            "prompt": prompt_text,
            "output": result_text,
            "usage":  usage_clean,
        })

        for cls in extract_answer(result):
            votes_per_class[cls] += 1

    selected_classes = [
        cls for cls in classes
        if votes_per_class[cls] > 0 and votes_per_class[cls] / num_agents >= consensus_threshold_ratio
    ]

    if not selected_classes:
        return "-", records
    return ", ".join(selected_classes), records


def bfs_search(
    table_name,
    table_in_markdown,
    column_name,
    ontology_dag,
    chains,
    model_names,
    max_depth,
    num_agents,
    consensus_threshold_ratio,
):
    """
    BFS over URI DAG; LLM only sees NAMES.
      - DAG structure remains URI-based: ontology_dag.root / ontology_dag.edges
      - At each level, candidate URIs are grouped by name locally (no global jump).
      - LLM selects names; we then expand all URIs under those names for next level.
      - search_path records names (since you only care about names).
    """
    edges    = ontology_dag.edges
    root_uri = ontology_dag.root

    queue = deque()
    queue.append((0, root_uri, []))

    possible_path = []
    all_records   = []

    while queue:
        level, parent_uri, search_path = queue.popleft()

        if level >= max_depth:
            possible_path.append(search_path)
            continue

        children_uris = edges.get(parent_uri, [])
        if not children_uris:
            possible_path.append(search_path)
            continue

        # group children URIs by name (local per-level mapping)
        name_groups        = ontology_dag.group_by_name(children_uris)
        current_level_names = sorted(name_groups.keys())

        print(f"Level {level}: {ontology_dag.name_of_uri(parent_uri)}")

        result, records = ensemble_decision_making(
            table_name,
            table_in_markdown,
            column_name,
            current_level_names,
            chains,
            model_names,
            num_agents,
            consensus_threshold_ratio,
        )
        all_records.extend(records)

        if result == "-":
            print("\tNone")
            possible_path.append(search_path)
            continue

        for sel_name in [x.strip() for x in result.split(",") if x.strip()]:
            if sel_name not in name_groups:
                print(f"Error: {sel_name} is not in current level ontology names.")
                continue

            print(f"\t{sel_name}")

            # expand all URIs under this name (you don't care which URI)
            for sel_uri in name_groups[sel_name]:
                queue.append((level + 1, sel_uri, search_path + [sel_name]))

    return possible_path, all_records

In [5]:
def run_single_experiment(model_names):
    chains     = [CHAIN_REGISTRY[name] for name in model_names]
    num_agents = len(model_names)
    consensus_threshold_ratio = THRESHOLD[num_agents]

    result_file_path = build_result_path(model_names)
    logs_path        = build_logs_path(model_names)
    summary_path     = build_summary_path(model_names)

    print("=" * 80)
    print(f"Running experiment: {build_experiment_tag(model_names)}")
    print("Models:            ", model_names)
    print("Threshold:         ", consensus_threshold_ratio)
    print("Result file:       ", result_file_path)
    print("Logs file:         ", logs_path)
    print("=" * 80)

    start_time = time.perf_counter()

    all_llm_records            = []
    semantic_annotation_results = []

    for index, row in df_labels.iterrows():
        table_id    = row["table_id"]
        column_name = row["column_name"]
        column_id   = int(row["column_id"])

        table             = dict_tables[table_id]
        table_name        = df_table_list[df_table_list["table_id"] == table_id]["table_name"].values[0]
        table_in_markdown = dataframe_to_markdown(table, k)

        print("+" * 70)
        print(table_id, table_name, column_id, column_name)

        paths, llm_records = bfs_search(
            table_name,
            table_in_markdown,
            column_name,
            dag,
            chains,
            model_names,
            max_depth=max_depth,
            num_agents=num_agents,
            consensus_threshold_ratio=consensus_threshold_ratio,
        )
        all_llm_records.extend(llm_records)
        semantic_annotation_results.append({
            "table_id":    table_id,
            "table_name":  table_name,
            "column_name": column_name,
            "column_id":   column_id,
            "paths":       paths,
        })

    end_time        = time.perf_counter()
    elapsed_time    = end_time - start_time
    elapsed_minutes = elapsed_time / 60

    token_summary = {
        "total":    {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0},
        "by_model": defaultdict(lambda: {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}),
    }

    for rec in all_llm_records:
        model = rec.get("model", "unknown_model")
        usage = rec.get("usage", {})
        for key in ["input_tokens", "output_tokens", "total_tokens"]:
            value = usage.get(key, 0) or 0
            token_summary["total"][key] += value
            token_summary["by_model"][model][key] += value

    summary = {
        "experiment": build_experiment_tag(model_names),
        "models":     model_names,
        "threshold":  consensus_threshold_ratio,
        "run_time": {
            "seconds": round(elapsed_time, 2),
            "minutes": round(elapsed_minutes, 2),
        },
        "token_usage_summary": {
            "total":    token_summary["total"],
            "by_model": dict(token_summary["by_model"]),
        },
    }

    with open(result_file_path, "w", encoding="utf-8") as f:
        json.dump(semantic_annotation_results, f, ensure_ascii=False, indent=2)

    with open(logs_path, "w", encoding="utf-8") as f:
        json.dump(all_llm_records, f, ensure_ascii=False, indent=2)

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print(f"Saved result  -> {result_file_path}")
    print(f"Saved logs    -> {logs_path}")
    print(f"Saved summary -> {summary_path}")

In [6]:
def run_all_experiments():
    for model_names in EXPERIMENTS:
        run_single_experiment(model_names=model_names)

run_all_experiments()

Running experiment: single_dsk
Models:             ['deepseek-chat']
Threshold:          0.66
Result file:        D:\RWTH\Github\sta-multi-llm-framework\results\main_edm\Result_edm\edm_single_dsk.json
Logs file:          D:\RWTH\Github\sta-multi-llm-framework\results\main_edm\Promptinput_edm\logs_edm_single_dsk.json
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
1.csv School building data.csv 0 Month
Level 0: Thing
	TemporalEntity
Level 1: TemporalEntity
	Interval
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
1.csv School building data.csv 1 Usage [kWh]
Level 0: Thing
	Measurement
Level 1: Measurement
	Energy unit
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
1.csv School building data.csv 2 Cost (net in PLN)
Level 0: Thing
	None
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
1.csv School building data.csv 3 Cost (gross in PLN)
Level 0: Thing
	None
Saved result  -> D:\RWTH\Github\sta-multi